# UC-ASSET-1 — Register a COG Asset and Upload via GCS

**Persona:** Data Engineer

**Goal:** Register a Cloud-Optimised GeoTIFF (COG) NDVI asset in a collection,
obtain a signed GCS `PUT` URL via `UploadTicket`, simulate the upload, poll the
status endpoint until `completed`, then verify the asset is queryable.

**Prerequisites:**
- GeoID platform running with the `GCPModule` active (provides `AssetUploadProtocol`)
- GCS bucket provisioned for the target catalog (Cloud Storage > Provisioning complete)
- Write-scoped JWT in env var `DYNASTORE_WRITE_TOKEN`
- Env vars: `DYNASTORE_BASE_URL`, `CATALOG_ID`, `COLLECTION_ID`

**Memory refs:**
- `project_geoid_storage_drivers` — GCP upload backend, `AssetUploadProtocol`
- `reference_geoid_api_surface` — `/assets/catalogs/{id}/collections/{cid}/upload`

In [ ]:
import os
import json
import time

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL     = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
WRITE_TOKEN  = os.environ.get("DYNASTORE_WRITE_TOKEN", "")
CATALOG_ID   = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "sentinel2-l2a")
GCS_BUCKET   = os.environ.get("GCS_BUCKET", "fao-geoid-assets")

# Asset ID must be unique; use a fixed name so teardown is deterministic
ASSET_ID     = "ndvi_2024_01_d1"
FILENAME     = "ndvi_2024_01_d1.tif"
CONTENT_TYPE = "image/tiff; application=geotiff"

headers = {
    "Authorization": f"Bearer {WRITE_TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30.0)
print(f"Connected to {BASE_URL}")
print(f"Catalog: {CATALOG_ID}  Collection: {COLLECTION_ID}  Bucket: {GCS_BUCKET}")

## Step 1 — Initiate upload

POST to `POST /assets/catalogs/{catalog_id}/collections/{collection_id}/upload`
with an `UploadRequest` body.  The GCP backend generates a signed resumable PUT URL
and returns an `UploadTicket` (status `pending`).  The ticket expires in ~1 hour.

In [ ]:
upload_payload = {
    "filename": FILENAME,
    "content_type": CONTENT_TYPE,
    "asset": {
        "asset_id": ASSET_ID,
        "asset_type": "RASTER",
        "title": "NDVI Dekad 1 Jan 2024",
        "roles": ["data"],
        "type": "image/tiff; application=geotiff",
        "metadata": {
            "variable": "ndvi",
            "period": "dekad",
            "dekad": 1,
            "year": 2024,
        },
    },
}

r = client.post(
    f"/assets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/upload",
    content=json.dumps(upload_payload),
)
print(r.status_code, r.text[:400])
assert r.status_code == 200, f"Expected 200 UploadTicket, got {r.status_code}: {r.text}"

ticket = r.json()
ticket_id  = ticket["ticket_id"]
upload_url = ticket["upload_url"]
method     = ticket["method"]       # "PUT" for GCS, "POST" for local
expires_at = ticket["expires_at"]
backend    = ticket["backend"]

print(f"\nTicket:     {ticket_id}")
print(f"Backend:    {backend}")
print(f"Method:     {method}")
print(f"Expires:    {expires_at}")
print(f"Upload URL: {upload_url[:80]}...")

## Step 2 — Upload to GCS (simulated)

In production the client sends the file bytes directly to the signed URL using
the `method` and `headers` from the ticket.  This notebook simulates that step
with a comment block to keep it runnable without file I/O.

```python
# Production upload — GCS backend:
# with open("ndvi_2024_01_d1.tif", "rb") as fh:
#     file_bytes = fh.read()
# upload_client = httpx.Client(timeout=300.0)
# resp = upload_client.request(
#     method=method,
#     url=upload_url,
#     content=file_bytes,
#     headers={"Content-Type": CONTENT_TYPE, **ticket.get("headers", {})},
# )
# assert resp.status_code in (200, 201), f"GCS upload failed: {resp.status_code}"
```

In [ ]:
# Simulate upload (no file bytes sent in this notebook)
print("Upload simulated — in production use:")
print(f"  {method} {upload_url[:80]}...")
print(f"  Content-Type: {CONTENT_TYPE}")
print()
print("For GCS backends: httpx.put(upload_url, content=file_bytes,")
print("    headers={'Content-Type': CONTENT_TYPE})")
print("For local backends: httpx.post(upload_url, files={'file': file_bytes})")
SIMULATED = True  # flag used by polling cell below

## Step 3 — Poll status until COMPLETED

`GET /assets/catalogs/{catalog_id}/upload/{ticket_id}/status` returns an
`UploadStatusResponse` with field `status` in
`{pending, uploading, completed, failed, cancelled}`.

For GCS backends the event is fired by `OBJECT_FINALIZE`; status transitions
are asynchronous.  For local backends status is `completed` immediately.
The simulated flag below skips the assertion so the notebook can run end-to-end
without a real file delivery.

In [ ]:
MAX_RETRIES = 10
POLL_INTERVAL_S = 2

attempts = 0
status_val = "pending"

while status_val not in ("completed", "failed", "cancelled") and attempts < MAX_RETRIES:
    r = client.get(
        f"/assets/catalogs/{CATALOG_ID}/upload/{ticket_id}/status"
    )
    assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"
    status_resp = r.json()
    status_val  = status_resp["status"]
    asset_id_registered = status_resp.get("asset_id")
    attempts += 1
    print(f"[attempt {attempts:2d}] status={status_val}  asset_id={asset_id_registered}")
    if status_val not in ("completed", "failed", "cancelled"):
        time.sleep(POLL_INTERVAL_S)

if SIMULATED:
    print("\nSimulated run: no file was delivered, so status remains 'pending' — expected.")
    print("In production this loop exits with status='completed' after the GCS event fires.")
else:
    assert status_val == "completed", (
        f"Upload did not complete within {MAX_RETRIES} retries. Final status: {status_val}"
    )
    print(f"\nAsset registered: {asset_id_registered}")

## Step 4 — Verify asset is queryable

After the GCS event completes and the asset is registered, retrieve it by ID
from the catalog-level assets endpoint.  A 200 response confirms the asset
record is persisted and queryable via the REST API.

**Note:** when running in simulated mode the asset was never actually registered,
so the GET returns 404.  In a real workflow this cell follows successful polling.

In [ ]:
r = client.get(f"/assets/catalogs/{CATALOG_ID}/assets/{ASSET_ID}")
print(r.status_code, r.text[:400])

if SIMULATED and r.status_code == 404:
    print("Simulated: asset not yet registered — expected 404 without real file delivery.")
else:
    assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"
    asset = r.json()
    print(f"\nhref  : {asset.get('uri')}")
    print(f"roles : {asset.get('metadata', {}).get('roles', 'n/a')}")
    print(f"type  : {asset.get('asset_type')}")
    assert asset["asset_id"] == ASSET_ID, "asset_id mismatch"
    assert asset["catalog_id"] == CATALOG_ID, "catalog_id mismatch"

## Edge cases

### UploadTicket TTL expired

The signed GCS URL is valid for approximately **1 hour** from ticket creation
(the exact expiry is in `ticket.expires_at`).  If the client takes longer to
deliver the file, the URL rejects the PUT with HTTP 403 (GCS) or 400 (S3).

**Re-initiate pattern** — simply call the upload endpoint again with the same
`asset_id`.  The platform checks `expires_at` before issuing the signed URL so
there is no need to clean up the old ticket manually.

In [ ]:
from datetime import datetime, timezone

expires_dt = datetime.fromisoformat(expires_at.replace("Z", "+00:00"))
now_utc    = datetime.now(tz=timezone.utc)
remaining  = (expires_dt - now_utc).total_seconds()

print(f"Ticket expires at : {expires_at}")
print(f"Seconds remaining : {remaining:.0f}s")

if remaining <= 0:
    print("\nTicket expired — re-initiating upload session...")
    r2 = client.post(
        f"/assets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/upload",
        content=json.dumps(upload_payload),
    )
    assert r2.status_code == 200, f"Re-initiation failed: {r2.status_code}: {r2.text}"
    new_ticket = r2.json()
    print(f"New ticket: {new_ticket['ticket_id']}  expires: {new_ticket['expires_at']}")
else:
    print("Ticket still valid — no re-initiation needed.")

In [ ]:
# Local backend fallback
# When the GCS module is not loaded the platform falls back to the local upload backend.
# The ticket method will be "POST" and upload_url will be a server-side proxy path.
#
# Local upload pattern:
#
#   ticket = r.json()
#   assert ticket["method"] == "POST"
#   assert ticket["backend"] == "local"
#
#   with open(FILENAME, "rb") as fh:
#       resp = httpx.post(
#           BASE_URL + ticket["upload_url"],
#           files={"file": (FILENAME, fh, CONTENT_TYPE)},
#           headers={"Authorization": f"Bearer {WRITE_TOKEN}"},
#       )
#   assert resp.status_code in (200, 201), f"Local upload failed: {resp.status_code}"
#   # Status is 'completed' immediately — no polling needed.

print(f"Current backend is '{backend}'.")
if backend == "local":
    print("Local backend active: upload via POST multipart; status is 'completed' immediately.")
else:
    print("GCS/S3 backend active: upload via signed PUT URL; poll status until 'completed'.")

## Teardown

Hard-delete the asset from the catalog.  This issues `force=true` which bypasses
the soft-delete and removes the row immediately.  Only run this after reviewing
the output above — the operation is irreversible.

In [ ]:
r = client.delete(
    f"/assets/catalogs/{CATALOG_ID}/assets/{ASSET_ID}",
    params={"force": "true"},
)
print(r.status_code, r.text[:200])
if r.status_code == 404:
    print("Asset not found (simulated run) — nothing to delete.")
else:
    assert r.status_code == 204, f"Expected 204, got {r.status_code}: {r.text}"
    print(f"Asset '{ASSET_ID}' hard-deleted from catalog '{CATALOG_ID}'.")
client.close()